In [20]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

#dataset na het opschonen van missing values inlezen
df = pd.read_csv('Leesmateriaal_DU.csv')
df = df.convert_dtypes()
df.dtypes

Titel                                string
Auteur                               string
Korte omschrijving                   string
Type materiaal                       string
Niveau                               string
genre                                string
onderwerp                            string
trefwoorden                          string
taal                                 string
aantal_paginas                        Int64
geschatte_leestijd_min                Int64
bron_url                             string
doelgroep                            string
gegevensstatus                       string
leescomplexiteit_score              Float64
interessecluster                     string
uitgever                             string
publicatiejaar                        Int64
beschikbaar_formaat                  string
cataloguscode                        string
leescomplexiteit_score_ontbreekt    boolean
dtype: object

In [21]:
def maak_zoektekst(df):
    kolommen = [
        "Korte omschrijving", "Type materiaal", "Niveau", "genre",
        "onderwerp", "trefwoorden", "taal", "doelgroep"
    ]

    zoektekst = df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return zoektekst


df = maak_zoektekst(df)

df.iloc[[0]]

0    Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. Boek 3F Fictie WOII WOII; onderduik Nederlands MBO
dtype: str

In [23]:
import re
import spacy

nlp = spacy.load("nl_core_news_lg")

In [24]:
df["tekst_lowercase"] = (
    df
    .str.lower()
    .str.strip()
)

df["tekst_lowercase"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam. haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. boek 3f fictie woii woii; onderduik nederlands mbo
dtype: str

In [25]:
def tekst_opschonen(tekst):
    # Verwijder losse cijfers, maar behoud bijvoorbeeld 2f en 3f
    tekst = re.sub(r"(?<![A-Za-zÀ-ÿ])\d+(?![A-Za-zÀ-ÿ])", " ", tekst)

    # Verwijder leestekens, maar behoud letters, cijfers en spaties
    tekst = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", tekst)

    # Maak meerdere spaties één spatie
    tekst = re.sub(r"\s+", " ", tekst)

    return tekst.strip()

df["tekst_schoon"] = df["tekst_lowercase"].apply(tekst_opschonen)

df["tekst_schoon"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam haar persoonlijke observaties geven een indringend beeld van angst hoop groei en het dagelijks leven onder vervolging boek 3f fictie woii woii onderduik nederlands mbo
dtype: str

In [26]:
df["tokens"] = df["tekst_schoon"].apply(
    lambda tekst: [token.text for token in nlp(tekst)]
)

df["tokens"].iloc[[0]]

0    [dagboek, van, een, joods, meisje, dat, tijdens, de, tweede, wereldoorlog, onderduikt, in, amsterdam, haar, persoonlijke, observaties, geven, een, indringend, beeld, van, angst, hoop, groei, en, het, dagelijks, leven, onder, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo]
dtype: object

In [27]:
df["tokens_zonder_stopwoorden"] = df["tekst_schoon"].apply(
    lambda tekst: [
        token.text
        for token in nlp(tekst)
        if not token.is_stop and token.is_alpha
    ]
)

df["tokens_zonder_stopwoorden"].iloc[[0]]

0    [dagboek, joods, meisje, tweede, wereldoorlog, onderduikt, amsterdam, persoonlijke, observaties, indringend, beeld, angst, hoop, groei, dagelijks, leven, vervolging, boek, fictie, woii, woii, onderduik, nederlands, mbo]
dtype: object

In [28]:
def lemmatiseer(tekst):
    doc = nlp(tekst)

    return [
        token.lemma_
        for token in doc
        if not token.is_stop and token.is_alpha
    ]

df["tokens_basisvorm"] = df["tekst_schoon"].apply(lemmatiseer)

df["tokens_basisvorm"].iloc[[0]]

0    [dagboek, joods, meisje, twee, wereldoorlog, onderduiken, Amsterdam, persoonlijk, observatie, indringen, beeld, angst, hopen, groei, dagelijks, leven, vervolging, boek, fictie, woii, woii, onderduiken, Nederlands, mbo]
dtype: object

In [36]:
df["tekst_nlp"] = df["tokens_basisvorm"].apply(
    lambda tokens: " ".join(tokens)
)

df["tekst_nlp"].iloc[0:3]

0    dagboek joods meisje twee wereldoorlog onderduiken Amsterdam persoonlijk observatie indringen beeld angst hopen groei dagelijks leven vervolging boek fictie woii woii onderduiken Nederlands mbo
1                                       spannend thriller waarin vrouw confronteren stalking bedreiging spanning oplopen zetten gezin beschermen boek fictie spanning spanning stalking Nederlands mbo
2                      toegankelijk verhaal liefde relatie zoektocht passen partner plattelandsleven vormen decor herkenbaar menselijk ervaring keuze boek fictie liefde liefde relatie Nederlands mbo
dtype: str

In [35]:
df.iloc[0:3]

0    Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. Boek 3F Fictie WOII WOII; onderduik Nederlands MBO
1               Spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. Terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. Boek 3F Fictie spanning spanning; stalking Nederlands MBO / NT2
2                                 Toegankelijk verhaal over liefde, relaties en de zoektocht naar een passende partner. Het plattelandsleven vormt het decor voor herkenbare menselijke ervaringen en keuzes. Boek 2F Fictie liefde liefde; relaties Nederlands MBO / NT2
dtype: object